# Tutorial 2: Phase analysis with tree search
Dara is equipped with a parallelized tree search algorithm to identify possible phases
present in a given XRD pattern.

In this tutorial, we will try to identify the phases in one experimental solid-state
reaction sample between `GeO2` and `ZnO`.

> You can download this tutorial project from [here](https://idocx.github.io/dara/_static/tutorial.zip).

In [ ]:
%pip install ipywidgets nbformat

In [ ]:
from pathlib import Path

from dara import search_phases

In [ ]:
pattern_path = "tutorial_data/GeO2-ZnO_700C_60min.xrdml"

# three elements are present in the sample
chemical_system = "Ge-O-Zn"

## Step 1: Prepare reference phases

Dara pre-builds an index of all the unique and low-energy phases in ICSD and COD
databases. It also implements a method to download CIF structures from COD data server
so that there is no need to obtain the offline database.

Before every search, we will need to gather all the reference phases in the chemical
system for the search algorithm. Dara provides `ICSDDatabase` and `CODDatabase` to do
the filtering.

In this example, we will use `CODDatabase` to download all the phases in the chemical system of `Ge-O-Zn`.

In [ ]:
from dara.structure_db import CODDatabase

# The COD database contains methods to filter phases in the chemical system
cod_database = CODDatabase()

# gather reference phases and save them to a directory called "cifs"
all_icsd_ids = cod_database.get_cifs_by_chemsys(chemical_system, dest_dir="cifs")

Since we are using a pre-filterd database (i.e., the COD), the downloaded CIF files will automatically be named according to the
following convention:

```
{composition}_{spacegroup}_(cod|icsd_{id})-{e_hull}.cif
```
Where the `e_hull` is the energy above the convex hull in meV/atom, as determined from
the Materials Project database for the ground-state entry with matching composition and spacegroup.

## Step 2: Search for phases

After preparing the reference CIFs, we can start the phase search on a provided XRD
pattern.

In this case, we are using the XRD pattern from the solid-state reaction sample
on our laboratory's Aeris diffractometer (`tutorial_data/GeO2-ZnO_700C_60min.xrdml`).

In [ ]:
# gather all the phases in the "cifs" directory
all_cifs = list(Path("cifs").glob("*.cif"))

search_results = search_phases(
    pattern_path=pattern_path,
    phases=all_cifs,
    wavelength="Cu",
    instrument_profile="Aeris-fds-Pixcel1d-Medipix3",
)

## Step 3: Result analysis
The returned search result will be a list of `SearchResult` object.

In [ ]:
search_results

In this pattern, we only have one solution found with `Rwp = 12.04 %`.

In [ ]:
for i in range(len(search_results)):
    print(f"Rwp of solution {i} = {search_results[i].refinement_result.lst_data.rwp} %")

Each `SearchResult` has a `.visualize()` method to visualize the refined pattern and
missing/extra peaks in the solution. If there are no missing or extra peaks, this option
will not appear.

In [ ]:
search_results[0].visualize()

You can also view all the alternative phases in one solution from `SearchResult.phases` attribute.

In [ ]:
print("Phases found in solution 0:")
for i, phases_ in enumerate(search_results[0].phases):
    print(f"    - Phase {i}: {[phase.path.name for phase in phases_]}")

From the result, you can see that for the phase `GeO2`, the algorithm identifies two
similar phases with slightly different spacegroups (152 and 154).